# Wfo Mag7 Sma Optimization

Interactive notebook version of this workflow. Edit the parameters and rerun cells to experiment.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / 'quant_orchestrator').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dataclasses import dataclass
from itertools import product
from time import perf_counter
from typing import Any

import pandas as pd

from quant_orchestrator.experiments import WindowSpec, build_walk_forward_windows
from quant_orchestrator.platforms.backtesting_frameworks.backtesting_py.data_adapter import (
    build_backtesting_frame,
)
from quant_orchestrator.platforms.backtesting_frameworks.nautilus.data_adapter import (
    build_nautilus_in_memory_data,
)
from quant_orchestrator.platforms.backtesting_frameworks.shared import (
    MAG7_SYMBOLS,
    load_price_frame,
    normalize_session_label,
)
from quant_orchestrator.platforms.backtesting_frameworks.zipline.data_adapter import (
    build_zipline_in_memory_data,
)
from quant_orchestrator.strategy import summarize_backtest
from quant_warehouse.feature_engineering import compute_features_worldclass


FAST_WINDOWS = (10, 20, 50)
SLOW_WINDOWS = (100, 150, 200)
CAPITAL_BASE = 100_000.0
SYMBOL_COUNT = len(MAG7_SYMBOLS)

WFO_SPEC = WindowSpec(
    mode="fixed",
    train_start="2020-01-01",
    train_end="2025-12-31",
    test_start="2026-01-01",
    test_end="2026-12-31",
)
TRAIN_WINDOW = build_walk_forward_windows(WFO_SPEC)[0]


@dataclass(frozen=True)
class FrameworkRun:
    raw_result: Any
    summary: pd.DataFrame
    equity: pd.Series


def _patch_zipline_compatibility() -> None:
    if getattr(_patch_zipline_compatibility, "_patched", False):
        return

    import zipline.finance.ledger as ledger_mod
    from zipline.data.in_memory_daily_bars import InMemoryDailyBarReader

    InMemoryDailyBarReader.frames = property(lambda self: self._frames)
    ledger_mod.PositionTracker.stats = property(lambda self: self._stats)
    _patch_zipline_compatibility._patched = True  # type: ignore[attr-defined]


def build_sma_frame(prices: pd.DataFrame, *, fast_window: int, slow_window: int) -> pd.DataFrame:
    if fast_window >= slow_window:
        raise ValueError('fast_window must be less than slow_window')

    frame = compute_features_worldclass(prices.copy())
    fast_col = f'SMA{fast_window}'
    slow_col = f'SMA{slow_window}'
    missing = [column for column in (fast_col, slow_col) if column not in frame.columns]
    if missing:
        raise ValueError(
            'Quant Warehouse feature output is missing required SMA columns: '
            f'{missing}. Update quant-warehouse feature engineering first.'
        )

    frame['fast_sma'] = frame[fast_col]
    frame['slow_sma'] = frame[slow_col]
    frame['signal'] = (frame['fast_sma'] > frame['slow_sma']).astype(int).fillna(0)
    frame = frame.dropna(subset=['open', 'high', 'low', 'close', 'volume']).copy()
    return frame



def _combine_equity_sleeves(sleeves: list[pd.Series]) -> pd.Series:
    combined_index = pd.Index([])
    for sleeve in sleeves:
        combined_index = combined_index.union(sleeve.index)
    combined = pd.Series(0.0, index=combined_index)
    for sleeve in sleeves:
        reindexed = sleeve.reindex(combined_index)
        reindexed = reindexed.ffill().fillna(sleeve.iloc[0])
        combined = combined.add(reindexed, fill_value=0.0)
    return combined.sort_index()


def run_backtesting_py(frame: pd.DataFrame, *, symbol: str, capital_base: float) -> FrameworkRun:
    from backtesting import Backtest, Strategy

    trade_size = max(1, int((capital_base * 0.25) / float(frame["close"].iloc[0])))
    bt_frame = build_backtesting_frame(frame).drop(columns=["Signal"])
    signal_map = {
        normalize_session_label(date): bool(signal)
        for date, signal in frame["signal"].items()
    }

    class SignalStrategy(Strategy):
        def init(self):
            self.trade_size = trade_size

        def next(self):
            bullish = signal_map.get(normalize_session_label(self.data.index[-1]), False)
            if bullish and not self.position:
                self.buy(size=self.trade_size)
            elif not bullish and self.position:
                self.position.close()

    started = perf_counter()
    stats = Backtest(
        bt_frame,
        SignalStrategy,
        cash=capital_base,
        commission=0.0,
        trade_on_close=False,
        exclusive_orders=True,
    ).run()
    elapsed = perf_counter() - started
    equity = stats["_equity_curve"]["Equity"].rename("portfolio_value")
    summary = summarize_backtest(
        framework="backtesting.py",
        symbol=symbol,
        equity=equity,
        elapsed_seconds=elapsed,
        bars=len(bt_frame),
        trades=int(stats["# Trades"]),
    )
    summary["native_return_pct"] = float(stats["Return [%]"])
    summary["native_sharpe"] = float(stats["Sharpe Ratio"]) if pd.notna(stats["Sharpe Ratio"]) else None
    summary["native_max_drawdown_pct"] = float(stats["Max. Drawdown [%]"])
    return FrameworkRun(stats, summary, equity)


def run_zipline(frame: pd.DataFrame, *, symbol: str, capital_base: float) -> FrameworkRun:
    from zipline.algorithm import TradingAlgorithm
    from zipline.api import order_target, record, symbol as zipline_symbol

    _patch_zipline_compatibility()
    adapter = build_zipline_in_memory_data(frame, symbol=symbol, capital_base=capital_base)
    trade_size = adapter.trade_size

    def initialize(context, **kwargs):
        context.asset = zipline_symbol(symbol.upper())
        context.is_long = False

    def handle_data(context, data):
        dt = normalize_session_label(context.get_datetime())
        bullish = adapter.signal_map.get(dt, False)
        if bullish and not context.is_long:
            order_target(context.asset, trade_size)
            context.is_long = True
        elif not bullish and context.is_long:
            order_target(context.asset, 0)
            context.is_long = False
        record(price=data.current(context.asset, "price"), signal=float(bullish))

    started = perf_counter()
    algo = TradingAlgorithm(
        sim_params=adapter.sim_params,
        data_portal=adapter.data_portal,
        asset_finder=adapter.asset_finder,
        initialize=initialize,
        handle_data=handle_data,
        capital_base=capital_base,
        benchmark_returns=adapter.benchmark_returns,
    )
    perf = algo.run()
    elapsed = perf_counter() - started
    equity = perf["portfolio_value"].rename("portfolio_value")
    transactions = perf.get("transactions", pd.Series(index=perf.index, data=[[]] * len(perf)))
    summary = summarize_backtest(
        framework="zipline",
        symbol=symbol,
        equity=equity,
        elapsed_seconds=elapsed,
        bars=len(perf),
        trades=int(transactions.map(len).sum()),
    )
    summary["native_last_value"] = float(perf["portfolio_value"].iloc[-1])
    return FrameworkRun(perf, summary, equity)


def run_nautilus(frame: pd.DataFrame, *, symbol: str, capital_base: float) -> FrameworkRun:
    from decimal import Decimal

    from nautilus_trader.backtest.engine import BacktestEngine
    from nautilus_trader.config import BacktestEngineConfig, LoggingConfig, StrategyConfig
    from nautilus_trader.model.currencies import USD
    from nautilus_trader.model.enums import AccountType, OmsType, OrderSide, TimeInForce
    from nautilus_trader.model.objects import Money, Quantity
    from nautilus_trader.trading.strategy import Strategy

    adapter = build_nautilus_in_memory_data(frame, symbol=symbol, capital_base=capital_base)

    class SignalStrategyConfig(StrategyConfig, frozen=True):
        instrument_id: object
        bar_type: object
        trade_size: Decimal
        signal_map: dict

    class SignalStrategy(Strategy):
        def __init__(self, config: SignalStrategyConfig):
            super().__init__(config)
            self.is_long = False

        def on_start(self) -> None:
            self.subscribe_bars(self.config.bar_type)

        def on_bar(self, bar) -> None:
            bullish = self.config.signal_map.get(normalize_session_label(bar.ts_event), False)
            if bullish == self.is_long:
                return
            side = OrderSide.BUY if bullish else OrderSide.SELL
            order = self.order_factory.market(
                instrument_id=self.config.instrument_id,
                order_side=side,
                quantity=Quantity.from_int(int(self.config.trade_size)),
                time_in_force=TimeInForce.IOC,
            )
            self.submit_order(order)
            self.is_long = bullish

    engine = BacktestEngine(
        config=BacktestEngineConfig(
            logging=LoggingConfig(log_level="OFF", bypass_logging=True),
        ),
    )
    engine.add_venue(
        venue=adapter.venue,
        oms_type=OmsType.NETTING,
        account_type=AccountType.CASH,
        starting_balances=[Money(capital_base, USD)],
        base_currency=USD,
    )
    engine.add_instrument(adapter.instrument)
    engine.add_data(adapter.bars)
    engine.add_strategy(
        SignalStrategy(
            SignalStrategyConfig(
                instrument_id=adapter.instrument.id,
                bar_type=adapter.bar_type,
                trade_size=Decimal(adapter.trade_size),
                signal_map=adapter.signal_map,
            ),
        ),
    )

    started = perf_counter()
    engine.run()
    fills_report = engine.trader.generate_order_fills_report()
    elapsed = perf_counter() - started
    engine.dispose()

    equity = _equity_from_fills(prices=frame, fills=fills_report, capital_base=capital_base)
    summary = summarize_backtest(
        framework="nautilus",
        symbol=symbol,
        equity=equity,
        elapsed_seconds=elapsed,
        bars=len(frame),
        trades=len(fills_report),
    )
    summary["native_fills"] = int(len(fills_report))
    summary["native_last_value"] = float(equity.iloc[-1])
    return FrameworkRun(fills_report, summary, equity)


def _equity_from_fills(*, prices: pd.DataFrame, fills: pd.DataFrame, capital_base: float) -> pd.Series:
    cash = float(capital_base)
    position = 0.0
    values = []
    fills_by_date: dict[pd.Timestamp, list[pd.Series]] = {}

    for _, fill in fills.iterrows():
        fill_date = normalize_session_label(fill["ts_last"])
        fills_by_date.setdefault(fill_date, []).append(fill)

    for date, row in prices.iterrows():
        normalized = normalize_session_label(date)
        for fill in fills_by_date.get(normalized, []):
            quantity = float(fill["filled_qty"])
            price = float(fill["avg_px"])
            if str(fill["side"]) == "BUY":
                cash -= quantity * price
                position += quantity
            else:
                cash += quantity * price
                position -= quantity
        values.append(cash + position * float(row["close"]))

    return pd.Series(values, index=prices.index, name="portfolio_value")


def _native_backtesting_sma_stats(
    *,
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    capital_base: float,
) -> tuple[pd.Series, dict[str, int]]:
    from backtesting import Backtest, Strategy
    from backtesting.lib import crossover

    def sma(values, window: int):
        return pd.Series(values).rolling(window).mean().to_numpy()

    def native_frame(frame: pd.DataFrame) -> pd.DataFrame:
        return frame.rename(
            columns={
                "open": "Open",
                "high": "High",
                "low": "Low",
                "close": "Close",
                "volume": "Volume",
            },
        ).copy()

    trade_size = max(1, int((capital_base * 0.25) / float(train_frame["close"].iloc[0])))

    class NativeSmaCross(Strategy):
        fast_window = 20
        slow_window = 100

        def init(self):
            self.fast = self.I(sma, self.data.Close, self.fast_window)
            self.slow = self.I(sma, self.data.Close, self.slow_window)

        def next(self):
            if crossover(self.fast, self.slow) and not self.position:
                self.buy(size=trade_size)
            elif crossover(self.slow, self.fast) and self.position:
                self.position.close()

    train_bt = Backtest(
        native_frame(train_frame),
        NativeSmaCross,
        cash=capital_base,
        commission=0.0,
        trade_on_close=False,
        exclusive_orders=True,
    )
    best_stats = train_bt.optimize(
        fast_window=FAST_WINDOWS,
        slow_window=SLOW_WINDOWS,
        maximize="Return [%]",
        constraint=lambda params: params.fast_window < params.slow_window,
    )
    best_fast = int(best_stats._strategy.fast_window)
    best_slow = int(best_stats._strategy.slow_window)

    test_stats = Backtest(
        native_frame(test_frame),
        NativeSmaCross,
        cash=capital_base,
        commission=0.0,
        trade_on_close=False,
        exclusive_orders=True,
    ).run(fast_window=best_fast, slow_window=best_slow)
    equity = test_stats["_equity_curve"]["Equity"].rename("portfolio_value")
    return equity, {"fast_window": best_fast, "slow_window": best_slow, "trades": int(test_stats["# Trades"])}


def _native_backtesting_portfolio(
    *,
    symbol_frames: dict[str, pd.DataFrame],
    capital_base: float,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    summaries = []
    sleeves = []
    capital_per_symbol = capital_base / SYMBOL_COUNT
    for symbol, frame in symbol_frames.items():
        train_frame = frame.loc[WFO_SPEC.train_start : WFO_SPEC.train_end]
        test_frame = frame.loc[WFO_SPEC.test_start : WFO_SPEC.test_end]
        started = perf_counter()
        equity, params = _native_backtesting_sma_stats(
            train_frame=train_frame,
            test_frame=test_frame,
            capital_base=capital_per_symbol,
        )
        summary = summarize_backtest(
            framework="backtesting.py-native",
            symbol=symbol,
            equity=equity,
            elapsed_seconds=perf_counter() - started,
            bars=len(equity),
            trades=int(params["trades"]),
        )
        summary["fast_window"] = params["fast_window"]
        summary["slow_window"] = params["slow_window"]
        summaries.append(summary)
        sleeves.append(equity.rename(symbol))

    portfolio_equity = _combine_equity_sleeves(sleeves)
    portfolio_summary = summarize_backtest(
        framework="backtesting.py-native",
        symbol="MAG7",
        equity=portfolio_equity,
        elapsed_seconds=sum(float(s["elapsed_seconds"].iloc[0]) for s in summaries),
        bars=len(portfolio_equity),
        trades=sum(int(s["trades"].iloc[0]) if pd.notna(s["trades"].iloc[0]) else 0 for s in summaries),
    )
    return pd.concat(summaries, ignore_index=True), portfolio_summary


def _run_portfolio_framework(
    framework: str,
    *,
    symbol_frames: dict[str, pd.DataFrame],
    capital_base: float,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, FrameworkRun]]:
    capital_per_symbol = capital_base / SYMBOL_COUNT
    runs: dict[str, FrameworkRun] = {}
    summaries = []
    sleeves = []
    elapsed = 0.0
    trades = 0

    for symbol, frame in symbol_frames.items():
        started = perf_counter()
        run = {
            "backtesting.py": run_backtesting_py,
            "zipline": run_zipline,
            "nautilus": run_nautilus,
        }[framework](frame, symbol=symbol, capital_base=capital_per_symbol)
        elapsed += perf_counter() - started
        trades += int(run.summary["trades"].iloc[0])
        summaries.append(run.summary)
        sleeves.append(run.equity.rename(symbol))
        runs[symbol] = run

    summary = pd.concat(summaries, ignore_index=True)
    portfolio_equity = _combine_equity_sleeves(sleeves)
    portfolio_summary = summarize_backtest(
        framework=framework,
        symbol="MAG7",
        equity=portfolio_equity,
        elapsed_seconds=elapsed,
        bars=len(portfolio_equity),
        trades=trades,
    )
    return summary, portfolio_summary, runs


def _build_symbol_frames(
    *,
    fast_window: int,
    slow_window: int,
    start: str,
    end: str,
    symbols: tuple[str, ...] = MAG7_SYMBOLS,
) -> dict[str, pd.DataFrame]:
    frames = {}
    for symbol in symbols:
        prices = load_price_frame(symbol, start=start, end=end)
        frames[symbol] = build_sma_frame(prices, fast_window=fast_window, slow_window=slow_window)
    return frames


def evaluate_candidate(
    *,
    framework: str,
    fast_window: int,
    slow_window: int,
    start: str,
    end: str,
) -> pd.DataFrame:
    symbol_frames = _build_symbol_frames(
        fast_window=fast_window,
        slow_window=slow_window,
        start=start,
        end=end,
    )
    _, portfolio_summary, _ = _run_portfolio_framework(
        framework,
        symbol_frames=symbol_frames,
        capital_base=CAPITAL_BASE,
    )
    portfolio_summary["fast_window"] = fast_window
    portfolio_summary["slow_window"] = slow_window
    return portfolio_summary


def optimize_framework(framework: str) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, FrameworkRun]]:
    train_rows = []
    for fast_window, slow_window in product(FAST_WINDOWS, SLOW_WINDOWS):
        if fast_window >= slow_window:
            continue
        train_rows.append(
            evaluate_candidate(
                framework=framework,
                fast_window=fast_window,
                slow_window=slow_window,
                start=WFO_SPEC.train_start,
                end=WFO_SPEC.train_end,
            ).iloc[0].to_dict()
        )

    grid = pd.DataFrame(train_rows).sort_values(
        by=["total_return", "max_drawdown", "final_equity"],
        ascending=[False, True, False],
    )
    best = grid.iloc[0]
    best_fast = int(best["fast_window"])
    best_slow = int(best["slow_window"])

    test_symbol_frames = _build_symbol_frames(
        fast_window=best_fast,
        slow_window=best_slow,
        start=WFO_SPEC.train_start,
        end=WFO_SPEC.test_end,
    )
    test_symbol_frames = {
        symbol: frame.loc[WFO_SPEC.test_start: WFO_SPEC.test_end].copy()
        for symbol, frame in test_symbol_frames.items()
    }
    _, test_portfolio_summary, test_runs = _run_portfolio_framework(
        framework,
        symbol_frames=test_symbol_frames,
        capital_base=CAPITAL_BASE,
    )
    test_portfolio_summary["fast_window"] = best_fast
    test_portfolio_summary["slow_window"] = best_slow
    test_portfolio_summary["train_best_total_return"] = float(best["total_return"])
    return grid, test_portfolio_summary, test_runs


def _print_native_snapshot(framework: str, raw_result) -> None:
    if framework == "backtesting.py":
        print(
            raw_result.loc[
                [
                    "Start",
                    "End",
                    "Equity Final [$]",
                    "Return [%]",
                    "Sharpe Ratio",
                    "Max. Drawdown [%]",
                    "# Trades",
                ]
            ]
        )
        return
    if framework == "zipline":
        print(raw_result.loc[:, ["portfolio_value", "returns"]].tail())
        return
    print(raw_result.loc[:, ["side", "filled_qty", "avg_px", "ts_last"]].head())


def main() -> None:
    print("Walk-forward schedule:")
    print(TRAIN_WINDOW)
    print()
    print("Native WFO support:")
    print(
        pd.DataFrame(
            [
                {"framework": "backtesting.py", "native_optimize": "yes"},
                {"framework": "zipline", "native_optimize": "no"},
                {"framework": "nautilus", "native_optimize": "no"},
            ]
        ).to_string(index=False)
    )
    print()

    train_results = []
    test_results = []
    selected_params = []
    test_runs_by_framework = {}

    for framework in ("backtesting.py", "zipline", "nautilus"):
        grid, test_summary, test_runs = optimize_framework(framework)
        train_results.append(grid.assign(framework=framework))
        test_results.append(test_summary.assign(framework=framework))
        selected_params.append(
            {
                "framework": framework,
                "fast_window": int(test_summary["fast_window"].iloc[0]),
                "slow_window": int(test_summary["slow_window"].iloc[0]),
                "train_best_total_return": float(test_summary["train_best_total_return"].iloc[0]),
            }
        )
        test_runs_by_framework[framework] = test_runs

    train_grid = pd.concat(train_results, ignore_index=True)
    test_summary = pd.concat(test_results, ignore_index=True)
    native_train_grid, native_portfolio_summary = _native_backtesting_portfolio(
        symbol_frames=_build_symbol_frames(
            fast_window=FAST_WINDOWS[0],
            slow_window=SLOW_WINDOWS[0],
            start=WFO_SPEC.train_start,
            end=WFO_SPEC.test_end,
        ),
        capital_base=CAPITAL_BASE,
    )

    print("Train grid results:")
    print(
        train_grid[
            [
                "framework",
                "fast_window",
                "slow_window",
                "final_equity",
                "total_return",
                "max_drawdown",
                "daily_vol",
            ]
        ].to_string(index=False)
    )
    print()
    print("Native backtesting.py optimize results:")
    print(
        native_train_grid[
            [
                "symbol",
                "fast_window",
                "slow_window",
                "final_equity",
                "total_return",
                "max_drawdown",
            ]
        ].to_string(index=False)
    )
    print()
    print("Selected parameters:")
    print(pd.DataFrame(selected_params).to_string(index=False))
    print()
    print("2026 test portfolio summary:")
    print(
        test_summary[
            [
                "framework",
                "symbol",
                "fast_window",
                "slow_window",
                "bars",
                "trades",
                "final_equity",
                "total_return",
                "max_drawdown",
                "elapsed_seconds",
            ]
        ].to_string(index=False)
    )
    print()
    print("Native backtesting.py portfolio summary:")
    print(
        native_portfolio_summary[
            [
                "framework",
                "symbol",
                "bars",
                "trades",
                "final_equity",
                "total_return",
                "max_drawdown",
                "elapsed_seconds",
            ]
        ].to_string(index=False)
    )
    print()
    print("Native engine snapshots:")
    for framework in ("backtesting.py", "zipline", "nautilus"):
        print(f"[{framework}]")
        for symbol in MAG7_SYMBOLS:
            print(f"=== {symbol} ===")
            _print_native_snapshot(framework, test_runs_by_framework[framework][symbol].raw_result)
            print()

    print(
        "Note: Zipline Reloaded and NautilusTrader do not expose a native walk-forward optimizer in this setup. "
        "Backtesting.py does, but it optimizes per symbol, so the notebook prints that comparison separately."
    )


if __name__ == "__main__":
    main()


Walk-forward schedule:
WalkForwardWindow(train_start=Timestamp('2020-01-01 00:00:00'), train_end=Timestamp('2025-12-31 00:00:00'), test_start=Timestamp('2026-01-01 00:00:00'), test_end=Timestamp('2026-12-31 00:00:00'))

Native WFO support:
     framework native_optimize
backtesting.py             yes
       zipline              no
      nautilus              no



/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/backtesting/_plotting.py:50: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support (e.g. PyCharm, Spyder IDE). Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(
/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbo

...

Backtest.optimize:   0%|          | 0/9 [00:00<?, ?it/s]

Backtest.optimize:   0%|          | 0/9 [00:00<?, ?it/s]

Backtest.optimize:   0%|          | 0/9 [00:00<?, ?it/s]

Backtest.optimize:   0%|          | 0/9 [00:00<?, ?it/s]

Train grid results:
     framework  fast_window  slow_window  final_equity  total_return  max_drawdown  daily_vol
backtesting.py           10          100     161614.07        0.6161       -0.1213     0.0058
backtesting.py           20          100     158977.19        0.5898       -0.1285     0.0059
backtesting.py           50          100     144065.81        0.4407       -0.1335     0.0054
backtesting.py           20          200     137901.26        0.3790       -0.1598     0.0058
backtesting.py           10          150     137207.19        0.3721       -0.1435     0.0057
backtesting.py           50          150     137044.29        0.3704       -0.1530     0.0060
backtesting.py           20          150     136381.57        0.3638       -0.1511     0.0058
backtesting.py           50          200     135973.81        0.3597       -0.1687     0.0059
backtesting.py           10          200     134297.47        0.3430       -0.1583     0.0055
       zipline           20          100